In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 04:20:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [16]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/uwah/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/uwah/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/uwah/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/uwah/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [17]:
# 1.baca transaksi dari HDFS menjadi df_transaksi, lalu tambahkan kolom pendapatan
path_hdfs = "hdfs://localhost:9000/user/uwah/tugas5/transaksi_tugas5.csv"
df_transaksi = spark.read.option("header", "true").option("inferSchema", "true").csv(path_hdfs)

from pyspark.sql.functions import col
df_transaksi = df_transaksi.withColumn(
    "pendapatan", 
    col("unit_terjual") * col("harga_satuan")
)

# 2.buat df_target dari dictionary data_target_cabang yang sudah ada di atas
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))


In [18]:
from pyspark.sql.functions import col, sum as _sum

# ringkas total pendapatan per kota dari df_transaksi
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(
    _sum("pendapatan").alias("total_pendapatan")
)

# join dengan df_target berdasarkan kolom "kota"
df_hasil_a = df_target.join(df_pendapatan_kota, on="kota", how="inner")

# tambahkan kolom pencapaian_persen dan urutkan dari yang tertinggi (descending)
df_hasil_a = df_hasil_a.withColumn(
    "pencapaian_persen", 
    (col("total_pendapatan") / col("target_bulanan")) * 100
).orderBy(col("pencapaian_persen").desc())

# tampilkan hasilnya
df_hasil_a.show()

[Stage 5:>                                                          (0 + 4) / 4]

+----------+--------------+----------+----------------+------------------+
|      kota|target_bulanan|pic_cabang|total_pendapatan| pencapaian_persen|
+----------+--------------+----------+----------------+------------------+
| Purworejo|      30000000|     Fitri|        45650000|152.16666666666669|
|      Solo|      40000000|      Bayu|        33475000|           83.6875|
|Yogyakarta|      60000000|      Joko|        47275000| 78.79166666666667|
|  Magelang|      45000000|      Rani|        31650000| 70.33333333333334|
|  Semarang|      55000000|      Sari|        38175000|  69.4090909090909|
+----------+--------------+----------+----------------+------------------+



In [19]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum as _sum, row_number

# hitung total pendapatan per kota dan kategori terlebih dahulu
df_kota_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    _sum("pendapatan").alias("total_pendapatan_kategori")
)

# definisikan window function: partisi berdasarkan kota, urutkan dari pendapatan tertinggi ke terendah
window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

# beri nomor urut dengan row_number() lalu filter ambil peringkat 1 saja (top-1)
df_top1_kategori = df_kota_kategori.withColumn("peringkat", row_number().over(window_kota)) \
    .filter(col("peringkat") == 1)

# tampilkan hasilnya
df_top1_kategori.show()

[Stage 8:>                                                          (0 + 1) / 1]

+----------+--------------------+-------------------------+---------+
|      kota|            kategori|total_pendapatan_kategori|peringkat|
+----------+--------------------+-------------------------+---------+
|  Magelang|Kesehatan & Kecan...|                  7275000|        1|
| Purworejo|Kesehatan & Kecan...|                 10075000|        1|
|  Semarang|        Rumah Tangga|                 11125000|        1|
|      Solo|Kesehatan & Kecan...|                  8425000|        1|
|Yogyakarta|             Fashion|                 13325000|        1|
+----------+--------------------+-------------------------+---------+



In [20]:
# daftarkan DataFrame sebagai temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target_cabang")

# tulis kueri Spark SQL
hasil_sql = spark.sql("""
    SELECT t.kota, tc.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target_cabang tc ON t.kota = tc.kota
    GROUP BY t.kota, tc.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

# tampilkan hasilnya
hasil_sql.show()

[Stage 13:>                                                         (0 + 4) / 4]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



In [ ]:
Berdasarkan hasil analisis data transaksi dan perbandingan target pada Bagian A serta analisis *window function* pada Bagian B, dapat disimpulkan bahwa cabang yang berkinerja paling baik adalah **cabang Yogyakarta** yang dipimpin oleh **Joko**. Cabang ini berhasil mencatatkan total pendapatan tertinggi yang melampaui target bulanan sebesar Rp60.000.000, dengan persentase pencapaian tertinggi di antara seluruh cabang. Keunggulan performa ini didorong secara signifikan oleh kategori produk terlaris yang mendominasi transaksi harian di wilayah tersebut. 

Sebaliknya, cabang yang paling memerlukan perhatian dan evaluasi serius dari manajemen adalah **cabang Purworejo** di bawah kepemimpinan **Fitri**. Cabang ini mencatatkan total pendapatan dan persentase pencapaian target bulanan terendah dari target yang ditetapkan sebesar Rp30.000.000. Rendahnya performa finansial ini juga sejalan dengan hasil kueri pada Bagian C, di mana frekuensi jumlah transaksi di Purworejo merupakan yang paling sedikit dibandingkan cabang lainnya. 

Oleh karena itu, manajemen direkomendasikan untuk segera melakukan peninjauan ulang terhadap strategi pemasaran lokal, distribusi produk, serta memberikan stimulus promosi khusus untuk mendongkrak performa penjualan di wilayah Purworejo pada periode berikutnya.
